# FIFA World Cup 2026: chronological modeling and toy simulation

This revised entry point uses tested modules. The original four-layer notebook is preserved at `notebooks/legacy/FIFA_World_Cup_2026_leaky.ipynb` for provenance and is not a valid evaluation. The source CSV contains contradictory mirrors; the audit quarantines them. No current Elo/FIFA/xG is attached to historical records.


In [ ]:
from pathlib import Path
import json
from wc2026.evaluate import run
from wc2026.simulation import Model, Scenario, TeamEvent, STAGES
from wc2026.export import export_matrix


## Four disjoint periods

Base training before 2023; held-out stacking in 2023; sigmoid calibration in 2024; untouched test from 2025 to the available data cutoff. Test features roll forward one day at a time using only earlier dated results. This is rolling one-step evaluation, not one frozen multi-year forecast. Mirror rows share dates/splits and are correlated observations.


In [ ]:
report, trained, history = run(Path("data/international_football_2018_present.csv"), Path("outputs/notebook"), seed=42)
report["partitions"]


In [ ]:
{name: {key: value for key, value in result.items() if key in ("log_loss", "multiclass_brier_sum", "accuracy", "top_label_ece")} for name, result in report["test_metrics"].items()}


## Interpretation

Read the full class-specific metrics and reliability bins in evaluation.json. A complex model is not assumed superior: compare all candidates with the Elo-only baseline. Accepted source records remain unverified against an authoritative match database. The small test period and correlated mirror rows limit conclusions. Do not label this retrospective exercise a validated World Cup forecast.


In [ ]:
audit = json.loads(Path("outputs/notebook/data-audit.json").read_text())
{key: audit[key] for key in ("source_rows", "accepted_matches", "rejected_groups")}


## Toy tournament

Group qualification uses a simplified ranking and Round-of-32 mapping. These are not official FIFA bracket odds. Static groups and Elo are unverified snapshots. Reproducible simulation does not imply accurate competition rules.


In [ ]:
baseline = Scenario(Model()).run(sims=200, seed=42)
sorted(((team, counts[5] / 200) for team, counts in baseline.items()), key=lambda row: -row[1])[:10]


In [ ]:
events = [TeamEvent("elo_shock", "France", delta=-100)]
scenario = Scenario(Model(), events).run(sims=200, seed=42)
# Same seed makes runs reproducible; events may alter random-number consumption.
# Differences still have Monte Carlo uncertainty.


## Export and parity

The calibrated held-out model can export symmetric loss/draw/win probabilities at the declared 2026-06-11 scenario date. Exporting does not validate the toy bracket. Run `python -m tools.compare_simulators --probs wc2026_prob_matrix.csv` after building Rust to compare exact unrounded counts.


In [ ]:
export_matrix(trained, history, "wc2026_prob_matrix.csv")
